# 2.1 — Exploratory Analysis & Label Joining

In [ ]:
import pandas as pd
import ipaddress

requests = pd.read_csv('http_requests.csv', parse_dates=['timestamp'])
labels = pd.read_csv('incident_labels.csv', parse_dates=['active_from', 'active_until', 'labeled_at'])

print(f'Requests: {requests.shape}')
print(f'Time range: {requests.timestamp.min()} → {requests.timestamp.max()}')
print(f'\nLabels: {labels.shape}')
print(labels.identifier_type.value_counts())
print()
print(labels.attack_class.value_counts())

In [ ]:
ip_labels = labels[labels.identifier_type == 'ip']
range_labels = labels[labels.identifier_type == 'ip_range']
tls_labels = labels[labels.identifier_type == 'tls_fingerprint']

# --- 1) IP exact match + temporal bounds ---
merged_ip = requests.merge(
    ip_labels[['source_identifier', 'attack_class', 'confidence', 'incident_id', 'active_from', 'active_until']],
    left_on='source_ip', right_on='source_identifier', how='inner'
)
merged_ip = merged_ip[
    (merged_ip.timestamp >= merged_ip.active_from) & (merged_ip.timestamp <= merged_ip.active_until)
]
print(f'IP exact matches: {len(merged_ip)}')

# --- 2) CIDR range match + temporal bounds ---
range_matches = []
for _, row in range_labels.iterrows():
    net = ipaddress.ip_network(row.source_identifier, strict=False)
    mask = requests.source_ip.apply(lambda x: ipaddress.ip_address(x) in net)
    temporal = (requests.timestamp >= row.active_from) & (requests.timestamp <= row.active_until)
    matched = requests[mask & temporal].copy()
    matched['incident_id'] = row.incident_id
    matched['attack_class'] = row.attack_class
    matched['confidence'] = row.confidence
    range_matches.append(matched)

merged_range = pd.concat(range_matches, ignore_index=True) if range_matches else pd.DataFrame()
print(f'CIDR range matches: {len(merged_range)}')

# --- 3) TLS fingerprint match + temporal bounds ---
merged_tls = requests.merge(
    tls_labels[['source_identifier', 'attack_class', 'confidence', 'incident_id', 'active_from', 'active_until']],
    left_on='tls_fingerprint', right_on='source_identifier', how='inner'
)
merged_tls = merged_tls[
    (merged_tls.timestamp >= merged_tls.active_from) & (merged_tls.timestamp <= merged_tls.active_until)
]
print(f'TLS fingerprint matches: {len(merged_tls)}')

In [ ]:
# Combine all malicious matches
cols = ['request_id', 'incident_id', 'attack_class', 'confidence']
all_malicious = pd.concat([
    merged_ip[cols],
    merged_range[cols] if len(merged_range) > 0 else pd.DataFrame(columns=cols),
    merged_tls[cols],
], ignore_index=True)

# Multi-incident requests
multi = all_malicious.groupby('request_id').incident_id.nunique()
print(f'Requests matched by multiple incidents: {(multi > 1).sum()}')

# Final labeling
malicious_ids = set(all_malicious.request_id)
requests['is_malicious'] = requests.request_id.isin(malicious_ids)

deduped = all_malicious.drop_duplicates(subset='request_id', keep='first')
requests = requests.merge(
    deduped[['request_id', 'attack_class', 'confidence']],
    on='request_id', how='left'
)

print(f'\nTotal requests: {len(requests)}')
print(f'Malicious: {requests.is_malicious.sum()} ({requests.is_malicious.mean()*100:.2f}%)')
print(f'Benign (unlabeled): {(~requests.is_malicious).sum()} ({(~requests.is_malicious).mean()*100:.2f}%)')
print(f'\nAttack class breakdown:')
print(requests[requests.is_malicious].attack_class.value_counts())
print(f'\nConfidence breakdown:')
print(requests[requests.is_malicious].confidence.value_counts())

In [ ]:
# Join method overlap analysis
ip_ids = set(merged_ip.request_id)
range_ids = set(merged_range.request_id) if len(merged_range) > 0 else set()
tls_ids = set(merged_tls.request_id)

print(f'Only IP match: {len(ip_ids - range_ids - tls_ids)}')
print(f'Only CIDR match: {len(range_ids - ip_ids - tls_ids)}')
print(f'Only TLS match: {len(tls_ids - ip_ids - range_ids)}')
print(f'IP ∩ CIDR: {len(ip_ids & range_ids)}')
print(f'IP ∩ TLS: {len(ip_ids & tls_ids)}')
print(f'CIDR ∩ TLS: {len(range_ids & tls_ids)}')

In [ ]:
# Labeling delay analysis
labels['labeled_at'] = pd.to_datetime(labels['labeled_at'], utc=True)
labels['delay_days'] = (labels.labeled_at - labels.active_until).dt.total_seconds() / 86400
print('Labeling delay (days from attack end to label):')
print(labels.groupby('attack_class')['delay_days'].agg(['mean', 'min', 'max']).round(2))

## Results & Discussion

### Label Counts

| | Count | % |
|---|---|---|
| Malicious | 592 | 1.18% |
| Benign (unlabeled) | 49,408 | 98.82% |

Attack breakdown: credential_stuffing (249), ddos_l7 (247), api_abuse (65), scanner (25), zero_day_exploit (6). Confidence: 521 high, 65 medium, 6 low.

### Labeling Gaps & Ambiguities

1. **Label absence ≠ benign.** The 49,408 unlabeled requests are assumed benign, but labels come from post-incident investigation — undetected attacks remain unlabeled. This contaminates the negative class.

2. **Labeling delay (0.1–2.9 days).** Labels arrive days after the attack. Zero-day exploits take ~3 days to label; DDoS ~0.5 day. Malicious traffic outside the `active_from/active_until` window but before `labeled_at` may be missed.

3. **532 requests matched by multiple incidents.** Mostly IP + TLS overlap (same source identified by both methods). When `attack_class` differs between incidents, choosing which label to keep is ambiguous — we took the first match.

4. **1.18% malicious vs. <0.1% constraint.** The synthetic dataset has ~12x more attack traffic than the stated production ratio. Class imbalance techniques and threshold tuning must account for this gap.

5. **CIDR ranges capture legitimate IPs.** A /24 block covers 256 IPs — labeling the entire range assumes all are compromised, which may introduce false positives in the ground truth itself.

6. **Shared TLS fingerprints.** The same JA3 hash can come from legitimate browsers with the same version/config as the attacker. TLS-based labels may incorrectly mark benign traffic.

7. **Missing `threat_intel_feed`.** The assessment mentions this table covering ~30% of attacks, but it was not provided. Label coverage is limited to incident-based labels only.